In [4]:
import sys
sys.argv = [
    "program",
    "--input_path", "/content/drive/MyDrive/Various-Model/data/sev.jsonl",
]


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"  # change to 7B later

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
).eval()

def generate(emotion, scenario, event):
    messages = [
        {
            "role": "system",
            "content": (
                f"You are a human experiencing {emotion}. "
                f"Write 1–2 natural sentences expressing this emotion clearly."
            )
        },
        {
            "role": "user",
            "content": f"Scenario: {scenario}\nEvent: {event}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.8,
            top_p=0.9
        )

    gen_text = tokenizer.decode(
        output[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return gen_text.strip()


# =======================
# TEST SAMPLES
# =======================

scenario = "I completed the project presentation and submitted it to the team."
event = "The team praised my work and decided to implement my ideas."

print("ANGER:")
print(generate("anger", scenario, event))
print("\nSADNESS:")
print(generate("sadness", scenario, event))


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


In [ ]:
# scripts/01_emotion_elicited_generation_prompt_based/1_emotion_elicited_generation_qwen.py
# -*- coding: utf-8 -*-
"""
情绪引导文本生成脚本 - Qwen版本
Emotion-Elicited Text Generation Script - Qwen Version

纯文本生成脚本：使用情绪引导 prompt 生成文本
Pure text generation script: Uses emotion-guided prompts to generate text

- 输入 Input: data/sev.jsonl 或 data/test_set.jsonl
- 输出 Output: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/generated/{dataset_name}_generated.jsonl
"""

import os, json, time, argparse
from pathlib import Path
from typing import Dict, Any

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

# HuggingFace Token
HF_TOKEN = ''  # Replace with your actual token
login(token=HF_TOKEN)

# 情绪类型列表 Emotion types
EMOTIONS = ["anger","sadness","happiness","fear","surprise","disgust"]
# 事件类型列表 Event types
VALENCES = ["positive","neutral","negative"]


def build_messages_for_emotion(emotion: str, scenario: str, event: str):
    """
    构建情绪引导的对话消息
    Build emotion-guided conversation messages
    """
    system = f"""
Always reply in {emotion}.
Keep the reply to at most two sentences.
""".strip()
    user = f"{scenario}\n{event}"
    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]


def iter_user_inputs(path: Path):
    """
    迭代读取输入文件
    Iterate through input file
    """
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            obj = json.loads(line)
            if "scenario" in obj and "event" in obj:
                yield obj


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_path", type=str, default=None,
                       help="输入数据路径 Input data path，如 e.g. data/sev.jsonl 或 data/test_set.jsonl")
    parser.add_argument("--both", action="store_true",
                       help="处理sev和test_set两个数据集 Process both sev and test_set datasets")
    parser.add_argument("--model", type=str, default="Qwen/Qwen3-4B-Instruct-2507",
                       help="模型名称 Model name (Qwen model)")
    parser.add_argument("--device", type=str, default="auto",
                       help="设备 Device")
    parser.add_argument("--dtype", type=str, default="bfloat16", choices=["float32","bfloat16","float16","auto"],
                       help="数据类型 Data type")
    parser.add_argument("--attn_impl", type=str, default="eager",
                       help="注意力实现 Attention implementation")
    parser.add_argument("--max_new_tokens", type=int, default=100,
                       help="最大生成token数 Max new tokens to generate")
    parser.add_argument("--seed", type=int, default=1234,
                       help="随机种子 Random seed")
    parser.add_argument("--valences", type=str, default="positive,neutral,negative",
                       help="效价列表 Valence list")
    parser.add_argument("--emotions", type=str, default="anger,sadness,happiness,fear,surprise,disgust",
                       help="情绪列表 Emotion list")
    parser.add_argument("--skip_if_exists", action="store_true", default=True,
                       help="跳过已处理的项目 Skip already processed items")
    parser.add_argument("--no_skip", action="store_true",
                       help="重新处理所有项目 Reprocess all items")
    args = parser.parse_args()

    # 确定输入文件列表
    # Determine input file list
    if args.both:
        input_paths = [Path("data/sev.jsonl"), Path("data/test_set.jsonl")]
    elif args.input_path:
        input_paths = [Path(args.input_path)]
    else:
        parser.error("必须指定 --input_path 或 --both | Must specify --input_path or --both")
        return

    # 从模型名称生成简化的文件夹名
    # Generate simplified folder name from model name
    # Qwen/Qwen2.5-7B-Instruct -> qwen25_7b
    model_name = args.model.split('/')[-1].lower()

    # Clean up the model name for folder
    model_folder = model_name.replace('-', '_').replace('.', '_')

    # 数据类型映射
    # Data type mapping
    if args.dtype == "auto":
        torch_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    else:
        dmap = {
            "float32": torch.float32,
            "bfloat16": torch.bfloat16,
            "float16": torch.float16
        }
        torch_dtype = dmap[args.dtype]

    torch.manual_seed(args.seed)

    # 加载分词器和模型（只加载一次）
    # Load tokenizer and model (only once)
    print(f"Loading Qwen model: {args.model}")
    print("This may take several minutes for larger models...")

    # Qwen specific: trust_remote_code is often required
    tok = AutoTokenizer.from_pretrained(
        args.model,
        use_fast=True,
        token=HF_TOKEN,
        trust_remote_code=True  # Qwen often requires this
    )

    # Set pad token if not set
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    # Load model with Qwen specific settings
    model = AutoModelForCausalLM.from_pretrained(
        args.model,
        torch_dtype=torch_dtype,
        device_map=args.device,
        attn_implementation=args.attn_impl,
        token=HF_TOKEN,
        trust_remote_code=True,  # Important for Qwen
        # Add quantization if needed for memory savings:
        # load_in_8bit=True,  # Uncomment for 8-bit quantization
        # load_in_4bit=True,  # Uncomment for 4-bit quantization
    )
    model.eval()
    print(f"Model loaded on device: {model.device}")
    print(f"Model dtype: {model.dtype}")
    print(f"Attention implementation: {args.attn_impl}\n")

    emotions = [e.strip() for e in args.emotions.split(",") if e.strip()]
    valences = [v.strip() for v in args.valences.split(",") if v.strip()]

    # 处理每个输入文件
    # Process each input file
    for data_path in input_paths:
        if not data_path.exists():
            print(f"[WARNING] Input file not found: {data_path}, skipping...")
            continue

        dataset_name = data_path.stem  # sev 或 test_set

        # 构建输出路径
        # Build output path: outputs/{model_folder}/01_emotion_elicited_generation_prompt_based/generated/{dataset_name}_generated.jsonl
        out_dir = Path("/content/drive/MyDrive/Various-Model/outputs") / model_folder / "01_emotion_elicited_generation_prompt_based" / "generated"
        out_dir.mkdir(parents=True, exist_ok=True)

        jsonl_path = out_dir / f"{dataset_name}_generated.jsonl"

        # 加载已存在的 keys（用于断点续跑）
        # Load existing keys (for resuming from checkpoint)
        existing_keys = set()
        if args.skip_if_exists and not args.no_skip and jsonl_path.exists():
            with open(jsonl_path, "r", encoding="utf-8") as f:
                for line in f:
                    try:
                        obj = json.loads(line.strip())
                        if "key" in obj:
                            existing_keys.add(obj["key"])
                    except:
                        continue

        print(f"{'='*70}")
        print(f"Processing dataset: {dataset_name}")
        print(f"Input: {data_path}")
        print(f"Output: {jsonl_path}")
        print(f"Model: {args.model}")
        print(f"{'='*70}\n")

        total = 0
        skipped = 0
        start = time.time()

        with open(jsonl_path, "a", encoding="utf-8") as fout:
            for item in iter_user_inputs(data_path):
                theme    = item.get("theme", "")
                scenario = item["scenario"]
                events   = item["event"]          # dict: positive/neutral/negative 事件字典
                sid      = item.get("skeleton_id", "unknown")

                for val in valences:
                    if val not in events: continue
                    event = events[val]

                    for emo in emotions:
                        key = f"{sid}__{val}__{emo}".replace("/", "_")

                        # 断点续跑检查
                        # Checkpoint resuming check
                        if key in existing_keys:
                            skipped += 1
                            if skipped % 50 == 0:
                                print(f"[SKIP] {skipped} items skipped so far... (last: {key})")
                            continue

                        # 构造 messages 并生成
                        # Build messages and generate
                        msgs = build_messages_for_emotion(emo, scenario, event)

                        # Apply chat template for Qwen
                        prompt = tok.apply_chat_template(
                            msgs,
                            tokenize=False,
                            add_generation_prompt=True
                        )

                        inputs = tok(prompt, return_tensors="pt").to(model.device)

                        # 贪婪生成
                        # Greedy generation
                        with torch.no_grad():
                            gen_ids = model.generate(
                                **inputs,
                                max_new_tokens=args.max_new_tokens,
                                do_sample=False,
                                top_p=None,
                                top_k=None,
                                temperature=None,
                                eos_token_id=tok.eos_token_id,
                                pad_token_id=tok.pad_token_id,
                                use_cache=True,
                            )
                        out_ids = gen_ids[0][inputs.input_ids.shape[1]:]
                        gen_text = tok.decode(out_ids, skip_special_tokens=True).strip()

                        # 保存到 jsonl
                        # Save to jsonl
                        row = {
                            "key": key,
                            "skeleton_id": sid,
                            "theme": theme,
                            "valence": val,
                            "emotion": emo,
                            "scenario": scenario,
                            "event": event,
                            "gen_text": gen_text,
                            "meta": {
                                "model_id": args.model,
                                "dtype": str(torch_dtype),
                                "device": str(model.device),
                                "attn_impl": args.attn_impl,
                                "max_new_tokens": args.max_new_tokens,
                                "seed": args.seed,
                                "input_len": int(inputs.input_ids.shape[1]),
                                "time": int(time.time()),
                            }
                        }
                        fout.write(json.dumps(row, ensure_ascii=False) + "\n")
                        fout.flush()  # 立即写入磁盘 Flush to disk immediately

                        total += 1
                        if total % 20 == 0:
                            el = time.time() - start
                            rate = total / el if el > 0 else 0
                            print(f"[progress] generated={total} | last={key} | {el:.1f}s elapsed | {rate:.2f} items/s")

        elapsed = time.time() - start
        print(f"\n[OK] {dataset_name} completed. Generated {total} items, skipped {skipped} items.")
        print(f"     Time: {elapsed:.1f}s | Rate: {total/elapsed:.2f} items/s")
        print(f"     Output: {jsonl_path}\n")


if __name__ == "__main__":
    main()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading Qwen model: Qwen/Qwen3-4B-Instruct-2507
This may take several minutes for larger models...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded on device: cuda:0
Model dtype: torch.bfloat16
Attention implementation: eager

Processing dataset: sev
Input: /content/drive/MyDrive/Various-Model/data/sev.jsonl
Output: /content/drive/MyDrive/Various-Model/outputs/qwen3_4b_instruct_2507/01_emotion_elicited_generation_prompt_based/generated/sev_generated.jsonl
Model: Qwen/Qwen3-4B-Instruct-2507

[SKIP] 50 items skipped so far... (last: work_02__negative__sadness)
[SKIP] 100 items skipped so far... (last: work_05__neutral__fear)
[SKIP] 150 items skipped so far... (last: work_08__positive__disgust)
[SKIP] 200 items skipped so far... (last: work_11__positive__sadness)
[SKIP] 250 items skipped so far... (last: work_13__negative__fear)
[SKIP] 300 items skipped so far... (last: work_16__neutral__disgust)
[SKIP] 350 items skipped so far... (last: work_19__neutral__sadness)
[SKIP] 400 items skipped so far... (last: school_02__positive__fear)
[SKIP] 450 items skipped so far... (last: school_04__negative__disgust)
[SKIP] 500 items s

In [6]:
import sys
sys.argv = [
    "program",
    "--input_path", "/content/drive/MyDrive/Various-Model/outputs/qwen3_4b_instruct_2507/01_emotion_elicited_generation_prompt_based/generated/sev_generated.jsonl",
]

In [8]:
 # scripts/01_emotion_elicited_generation_prompt_based/2_label_generated_with_gpt.py
# -*- coding: utf-8 -*-
"""
Emotion Text Labeling Script

Batch labeling script: Reads texts generated by script 1, uses GPT-4o-mini to label, saves by accepted/rejected

- Input: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/generated/{dataset_name}_generated.jsonl
- Output: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/labeled/{dataset_name}/{accepted|rejected}.jsonl
"""

import os, json, time, argparse
from pathlib import Path
from typing import Dict, Any

from openai import OpenAI

# ====== OpenAI (GPT-4o-mini) ======
client = OpenAI()

EMOTIONS = ["anger","sadness","happiness","fear","surprise","disgust"]

SYSTEM_LBL = f'''
You are a careful rater.
Given a target emotion and a text,
decide if the text's STYLE matches the target emotion among:
{EMOTIONS}
Focus on tone/attitude, not content valence.
'''.strip()

USER_TMPL_LBL = '''
Target emotion: {emotion}
Text:
{text}
Decide if the text's STYLE matches the target emotion.
Return a compact JSON with keys exactly:
{{
"match": <0 or 1>,
"reason": <short string>
}}
'''.strip()


def extract_json_from_response(response: str) -> str:
    """
    从GPT响应中提取JSON内容，处理markdown格式
    Extract JSON content from GPT response, handling markdown format
    """
    response = response.strip()

    # If contains markdown code block, extract JSON from it
    if "```json" in response:
        start = response.find("```json") + 7
        end = response.find("```", start)
        if end != -1:
            return response[start:end].strip()
    elif "```" in response:
        # Handle code blocks without json identifier
        start = response.find("```") + 3
        end = response.find("```", start)
        if end != -1:
            return response[start:end].strip()

    # If no code block, return original content
    return response


def ask_llm_label(client, model: str, emotion: str, text: str,
                  max_retries=4, backoff=1.8) -> Dict[str, Any]:
    """
    Call GPT-4o-mini for labeling; return unlabeled if no KEY or error
    """
    for i in range(max_retries):
        try:
            user_content = USER_TMPL_LBL.format(emotion=emotion, text=text)
            try:
                resp = client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": SYSTEM_LBL},
                        {"role": "user",   "content": user_content}
                    ],
                )
            except Exception as api_error:
                if i == max_retries - 1:
                    return {"match": 0, "reason": f"api-error:{type(api_error).__name__}"}
                time.sleep(backoff ** i)
                continue

            # Safely get response content
            try:
                choice = resp.choices[0]
                message = choice.message
                content = message.content
                out = (content or "").strip()
            except (KeyError, IndexError, AttributeError) as ke:
                return {"match": 0, "reason": f"response-structure-error: {type(ke).__name__}"}

            js = extract_json_from_response(out)
            try:
                j = json.loads(js)
                if "match" not in j: j = {"match": 0, "reason": "invalid-json"}
                if "reason" not in j: j["reason"] = "no-reason-provided"
                return j
            except json.JSONDecodeError as je:
                return {"match": 0, "reason": f"json-decode-error: {str(je)}"}
        except Exception as e:
            if i == max_retries - 1:
                return {"match": 0, "reason": f"error:{type(e).__name__}"}
            time.sleep(backoff ** i)

    return {"match": 0, "reason": "max-retries-exceeded"}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_path", type=str, default=None,
                       help="Input data path，如 outputs/llama32_3b/01_emotion_elicited_generation_prompt_based/generated/sev_generated.jsonl")
    parser.add_argument("--both", action="store_true",
                       help="sev test_set Process both sev and test_set datasets")
    parser.add_argument("--model_name", type=str, default="llama32_3b",
                       help=" Model folder name")
    parser.add_argument("--lbl_model", type=str, default="gpt-4o-mini",
                       help="Label model")
    parser.add_argument("--skip_if_exists", action="store_true", default=True,
                       help=" Skip already labeled items")
    parser.add_argument("--no_skip", action="store_true",
                       help=" Relabel all items")
    args = parser.parse_args()

    # Determine input file list
    if args.both:
        base_path = Path("/content/drive/MyDrive/Various-Model/outputs") / args.model_name / "01_emotion_elicited_generation_prompt_based" / "generated"
        input_paths = [
            base_path / "sev_generated.jsonl",
            base_path / "test_set_generated.jsonl"
        ]
    elif args.input_path:
        input_paths = [Path(args.input_path)]
    else:
        parser.error("必须指定 --input_path 或 --both | Must specify --input_path or --both")
        return

    # Process each input file
    for input_path in input_paths:
        if not input_path.exists():
            print(f"[WARNING] Input file not found: {input_path}, skipping...")
            continue

        # Extract model_name and dataset_name from input path
        parts = input_path.parts
        if "outputs" in parts and "01_emotion_elicited_generation_prompt_based" in parts and "generated" in parts:
            outputs_idx = parts.index("outputs")
            model_name = parts[outputs_idx + 1]
            filename = input_path.stem
            if filename.endswith("_generated"):
                dataset_name = filename[:-10]
            else:
                dataset_name = filename
        else:
            print(f"[ERROR] Input path format incorrect: {input_path}")
            print(f"        Expected: outputs/{{model_name}}/01_emotion_elicited_generation_prompt_based/generated/{{dataset_name}}_generated.jsonl")
            continue

        # Build output path
        out_dir = Path("/content/drive/MyDrive/Various-Model/outputs") / model_name / "01_emotion_elicited_generation_prompt_based" / "labeled" / dataset_name
        out_dir.mkdir(parents=True, exist_ok=True)

        acc_path = out_dir / "accepted.jsonl"
        rej_path = out_dir / "rejected.jsonl"

        # Load existing keys (for resuming from checkpoint)
        existing_keys = set()
        if args.skip_if_exists and not args.no_skip:
            for path in [acc_path, rej_path]:
                if path.exists():
                    with open(path, "r", encoding="utf-8") as f:
                        for line in f:
                            try:
                                obj = json.loads(line.strip())
                                if "key" in obj:
                                    existing_keys.add(obj["key"])
                            except:
                                continue

        print(f"{'='*70}")
        print(f"Processing dataset: {dataset_name}")
        print(f"Input: {input_path}")
        print(f"Output: {out_dir}")
        print(f"{'='*70}\n")

        total = 0
        skipped = 0
        accepted = 0
        rejected = 0
        start = time.time()

        # Statistics dictionaries: by emotion and valence
        stats_by_emotion = {}  # {emotion: {"total": N, "accepted": M}}
        stats_by_valence = {}  # {valence: {"total": N, "accepted": M}}

        with open(input_path, "r", encoding="utf-8") as fin:
            for line in fin:
                line = line.strip()
                if not line: continue

                try:
                    item = json.loads(line)
                except json.JSONDecodeError:
                    print(f"[WARN] Failed to parse line: {line[:100]}...")
                    continue

                key = item.get("key", "unknown")

                # Checkpoint resuming check
                if key in existing_keys:
                    skipped += 1
                    if skipped % 50 == 0:
                        print(f"[SKIP] {skipped} items skipped so far... (last: {key})")
                    continue

                emotion = item.get("emotion", "")
                valence = item.get("valence", "")
                gen_text = item.get("gen_text", "")

                # GPT labeling
                label = {"match": 0, "reason": "empty-text"}
                if isinstance(gen_text, str) and gen_text:
                    label = ask_llm_label(client, args.lbl_model, emotion, gen_text)

                # Add labeling result and timestamp
                item["judge"] = label
                item["label_time"] = int(time.time())

                # Save based on labeling result
                match_score = int(label.get("match", 0))
                if match_score == 1:
                    output_path = acc_path
                    accepted += 1
                    category = "accepted"
                else:
                    output_path = rej_path
                    rejected += 1
                    category = "rejected"

                with open(output_path, "a", encoding="utf-8") as fout:
                    fout.write(json.dumps(item, ensure_ascii=False) + "\n")

                # Update statistics
                if emotion:
                    if emotion not in stats_by_emotion:
                        stats_by_emotion[emotion] = {"total": 0, "accepted": 0}
                    stats_by_emotion[emotion]["total"] += 1
                    stats_by_emotion[emotion]["accepted"] += match_score

                if valence:
                    if valence not in stats_by_valence:
                        stats_by_valence[valence] = {"total": 0, "accepted": 0}
                    stats_by_valence[valence]["total"] += 1
                    stats_by_valence[valence]["accepted"] += match_score

                total += 1
                if total % 10 == 0:
                    el = time.time() - start
                    rate = total / el if el > 0 else 0
                    print(f"[progress] labeled={total} (acc={accepted}, rej={rejected}) | last={key} [{category}] | {el:.1f}s | {rate:.2f} items/s")

        elapsed = time.time() - start
        print(f"\n[OK] {dataset_name} completed. Labeled {total} items, skipped {skipped} items.")
        print(f"     Accepted: {accepted} | Rejected: {rejected}")
        print(f"     Time: {elapsed:.1f}s | Rate: {total/elapsed:.2f} items/s")
        print(f"     Output:")
        print(f"       - {acc_path}")
        print(f"       - {rej_path}")

        # ===== Output statistics =====
        print("\n" + "="*60)
        print(f"📊 ACCURACY STATISTICS - {dataset_name.upper()}")
        print("="*60)

        # Overall accuracy
        overall_acc = (accepted / total * 100) if total > 0 else 0
        print(f"\n🎯 Overall Accuracy: {accepted}/{total} = {overall_acc:.2f}%")

        # Statistics by emotion
        print(f"\n📈 Accuracy by Emotion:")
        print("-" * 60)
        print(f"{'Emotion':<15} {'Accepted':<12} {'Total':<10} {'Accuracy':<12}")
        print("-" * 60)

        emotions_sorted = sorted(stats_by_emotion.items())
        for emotion, stats in emotions_sorted:
            acc_count = stats["accepted"]
            tot_count = stats["total"]
            acc_rate = (acc_count / tot_count * 100) if tot_count > 0 else 0
            print(f"{emotion:<15} {acc_count:<12} {tot_count:<10} {acc_rate:>6.2f}%")

        # Statistics by valence
        print(f"\n📉 Accuracy by Valence:")
        print("-" * 60)
        print(f"{'Valence':<15} {'Accepted':<12} {'Total':<10} {'Accuracy':<12}")
        print("-" * 60)

        valences_sorted = sorted(stats_by_valence.items())
        for valence, stats in valences_sorted:
            acc_count = stats["accepted"]
            tot_count = stats["total"]
            acc_rate = (acc_count / tot_count * 100) if tot_count > 0 else 0
            print(f"{valence:<15} {acc_count:<12} {tot_count:<10} {acc_rate:>6.2f}%")

        print("="*60 + "\n")


if __name__ == "__main__":
    main()



Processing dataset: sev
Input: /content/drive/MyDrive/Various-Model/outputs/qwen3_4b_instruct_2507/01_emotion_elicited_generation_prompt_based/generated/sev_generated.jsonl
Output: /content/drive/MyDrive/Various-Model/outputs/qwen3_4b_instruct_2507/01_emotion_elicited_generation_prompt_based/labeled/sev

[progress] labeled=10 (acc=10, rej=0) | last=work_00__neutral__fear [accepted] | 15.9s | 0.63 items/s
[progress] labeled=20 (acc=20, rej=0) | last=work_01__positive__sadness [accepted] | 25.7s | 0.78 items/s
[progress] labeled=30 (acc=30, rej=0) | last=work_01__neutral__disgust [accepted] | 38.0s | 0.79 items/s
[progress] labeled=40 (acc=40, rej=0) | last=work_02__positive__fear [accepted] | 46.1s | 0.87 items/s
[progress] labeled=50 (acc=50, rej=0) | last=work_02__negative__sadness [accepted] | 57.9s | 0.86 items/s
[progress] labeled=60 (acc=60, rej=0) | last=work_03__positive__disgust [accepted] | 67.7s | 0.89 items/s
[progress] labeled=70 (acc=69, rej=1) | last=work_03__negative__fe

In [9]:
import sys
sys.argv = [
    "program",
    "--input_dir", "/content/drive/MyDrive/Various-Model/outputs/qwen3_4b_instruct_2507/01_emotion_elicited_generation_prompt_based/labeled",
    "--dataset", "sev"
]


In [10]:
# scripts/01_emotion_elicited_generation_prompt_based/3_generate_accuracy_stats.py
"""
Accuracy Statistics Generation Script

Reads accepted and rejected files from script 2, calculates accuracy by emotion and valence

-  Input: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/labeled/{dataset_name}/{accepted|rejected}.jsonl
-  Output: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/labeled/{dataset_name}/accuracy_stats.json
"""

import json, argparse
from pathlib import Path
from collections import defaultdict


def generate_accuracy_stats(labeled_dir: Path, dataset_name: str):
    """
    Generate accuracy statistics for specified dataset
    """

    dataset_dir = labeled_dir / dataset_name
    accepted_path = dataset_dir / "accepted.jsonl"
    rejected_path = dataset_dir / "rejected.jsonl"
    stats_path = dataset_dir / "accuracy_stats.json"

    if not dataset_dir.exists():
        print(f"[ERROR] Dataset directory not found: {dataset_dir}")
        return None

    # Statistics dictionaries
    stats_by_emotion = defaultdict(lambda: {"total": 0, "accepted": 0, "rejected": 0})
    stats_by_valence = defaultdict(lambda: {"total": 0, "accepted": 0, "rejected": 0})

    total_accepted = 0
    total_rejected = 0

    # Read accepted
    if accepted_path.exists():
        print(f"Reading {accepted_path}...")
        with open(accepted_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    item = json.loads(line)
                    emotion = item.get("emotion", "unknown")
                    valence = item.get("valence", "unknown")

                    stats_by_emotion[emotion]["total"] += 1
                    stats_by_emotion[emotion]["accepted"] += 1

                    stats_by_valence[valence]["total"] += 1
                    stats_by_valence[valence]["accepted"] += 1

                    total_accepted += 1
                except json.JSONDecodeError:
                    continue
    else:
        print(f"[WARNING] Accepted file not found: {accepted_path}")

    # Read rejected
    if rejected_path.exists():
        print(f"Reading {rejected_path}...")
        with open(rejected_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    item = json.loads(line)
                    emotion = item.get("emotion", "unknown")
                    valence = item.get("valence", "unknown")

                    stats_by_emotion[emotion]["total"] += 1
                    stats_by_emotion[emotion]["rejected"] += 1

                    stats_by_valence[valence]["total"] += 1
                    stats_by_valence[valence]["rejected"] += 1

                    total_rejected += 1
                except json.JSONDecodeError:
                    continue
    else:
        print(f"[WARNING] Rejected file not found: {rejected_path}")

    # Calculate accuracy
    for emotion in stats_by_emotion:
        total = stats_by_emotion[emotion]["total"]
        accepted = stats_by_emotion[emotion]["accepted"]
        stats_by_emotion[emotion]["accuracy"] = round(accepted / total * 100, 2) if total > 0 else 0.0

    for valence in stats_by_valence:
        total = stats_by_valence[valence]["total"]
        accepted = stats_by_valence[valence]["accepted"]
        stats_by_valence[valence]["accuracy"] = round(accepted / total * 100, 2) if total > 0 else 0.0

    # Overall statistics
    total_samples = total_accepted + total_rejected
    overall_accuracy = round(total_accepted / total_samples * 100, 2) if total_samples > 0 else 0.0

    # Build statistics result
    stats = {
        "dataset": dataset_name,
        "overall": {
            "total_samples": total_samples,
            "accepted": total_accepted,
            "rejected": total_rejected,
            "accuracy": overall_accuracy
        },
        "by_emotion": dict(stats_by_emotion),
        "by_valence": dict(stats_by_valence)
    }

    # Save statistics file
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    return stats, stats_path


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_dir", type=str, default=None,
                       help="labeled Labeled directory path outputs/llama32_3b/01_emotion_elicited_generation_prompt_based/labeled")
    parser.add_argument("--both", action="store_true",
                       help="sev test_set Process both sev and test_set datasets")
    parser.add_argument("--model_name", type=str, default="llama32_3b",
                       help="Model folder name")
    parser.add_argument("--dataset", type=str, default=None,
                       help="Dataset name (sev, test_set)")
    args = parser.parse_args()

    # Determine labeled directory path
    if args.both or (not args.input_dir and not args.dataset):
        labeled_dir = Path("/content/drive/MyDrive/Various-Model/outputs") / args.model_name / "01_emotion_elicited_generation_prompt_based" / "labeled"

    elif args.input_dir:
        labeled_dir = Path(args.input_dir)
    else:
        parser.error(" --input_dir --both | Must specify --input_dir or --both")
        return

    if not labeled_dir.exists():
        print(f"[ERROR] Labeled directory not found: {labeled_dir}")
        return

    # Determine datasets to process
    if args.both:
        datasets = ["sev", "test_set"]
    elif args.dataset:
        datasets = [args.dataset]
    else:
        # Auto-discover all dataset folders
        datasets = [d.name for d in labeled_dir.iterdir() if d.is_dir()]

    if not datasets:
        print(f"[ERROR] No datasets found in {labeled_dir}")
        return

    print("=" * 70)
    print("Generating Accuracy Statistics")
    print("=" * 70)

    for dataset_name in datasets:
        result = generate_accuracy_stats(labeled_dir, dataset_name)

        if result:
            stats, stats_path = result

            print(f"\n📊 {dataset_name.upper()} :")
            print(f"   File: {stats_path}")
            print(f"   Total: {stats['overall']['total_samples']}")
            print(f"   Accepted: {stats['overall']['accepted']}")
            print(f"   Rejected: {stats['overall']['rejected']}")
            print(f"   Overall Accuracy: {stats['overall']['accuracy']}%")

            print(f"\n   By Emotion:")
            for emotion in sorted(stats['by_emotion'].keys()):
                e_stats = stats['by_emotion'][emotion]
                print(f"      {emotion:12s}: {e_stats['accepted']:4d}/{e_stats['total']:4d} = {e_stats['accuracy']:6.2f}%")

            if stats['by_valence']:
                print(f"\n   By Valence:")
                for valence in sorted(stats['by_valence'].keys()):
                    v_stats = stats['by_valence'][valence]
                    print(f"      {valence:12s}: {v_stats['accepted']:4d}/{v_stats['total']:4d} = {v_stats['accuracy']:6.2f}%")

    print("\n" + "=" * 70)
    print("✅ Statistics generation completed!")
    print("=" * 70)


if __name__ == "__main__":
    main()

Generating Accuracy Statistics
Reading /content/drive/MyDrive/Various-Model/outputs/qwen3_4b_instruct_2507/01_emotion_elicited_generation_prompt_based/labeled/sev/accepted.jsonl...
Reading /content/drive/MyDrive/Various-Model/outputs/qwen3_4b_instruct_2507/01_emotion_elicited_generation_prompt_based/labeled/sev/rejected.jsonl...

📊 SEV :
   File: /content/drive/MyDrive/Various-Model/outputs/qwen3_4b_instruct_2507/01_emotion_elicited_generation_prompt_based/labeled/sev/accuracy_stats.json
   Total: 2880
   Accepted: 2804
   Rejected: 76
   Overall Accuracy: 97.36%

   By Emotion:
      anger       :  480/ 480 = 100.00%
      disgust     :  480/ 480 = 100.00%
      fear        :  476/ 480 =  99.17%
      happiness   :  479/ 480 =  99.79%
      sadness     :  472/ 480 =  98.33%
      surprise    :  417/ 480 =  86.88%

   By Valence:
      negative    :  916/ 960 =  95.42%
      neutral     :  944/ 960 =  98.33%
      positive    :  944/ 960 =  98.33%

✅ Statistics generation completed!
